# Import Packages

In [1]:
import os
import re
import logging
import numpy as np
from io import StringIO
from sklearn.metrics import confusion_matrix, classification_report
from IPython.display import clear_output

from bci_essentials.io.xdf_sources import XdfEegSource, XdfMarkerSource
from bci_essentials.bci_controller import BciController
from bci_essentials.paradigm.ssvep_paradigm import SsvepParadigm
from bci_essentials.data_tank.data_tank import DataTank
from bci_essentials.utils.logger import Logger  # Logger wrapper
from bci_essentials.classification.ssvep_fbcca_classifier import (
    SsvepFbCcaClassifier,
)

# Import Data & Initialize Objects

In [2]:
filename = os.path.join("data", "sub-Eli_ses-S003_task-T1_run-001_eeg.xdf")

logger = Logger(name=__name__)
logger.setLevel(1)
eeg_source = XdfEegSource(filename)
marker_source = XdfMarkerSource(filename)
paradigm = SsvepParadigm(live_update=False, iterative_training=False) # Setting live_upadate to False means it will classify each trial rather than each epoch (full 5 seconds rather than every second 5 time)
data_tank = DataTank()

2025-04-09 15:51:22 - INFO - __main__ : Setting logging level to Unknown Level


# Define the classifier

In [3]:
# Settings
target_frequencies = np.array([7.692307, 10, 12.5, 14.28571])
n_harmonics = 3  # Number of harmonics to use for SSVEP classification
fb_cutoffs = np.array([[i, 27] for i in range(3, 25, 2)])   # Filter bank cut-off frequencies
filter_order = 6
nsamples = 5 * 256  # 5 seconds of data at 256 Hz
fsample = 256.0  # Hz
concatenate_trials = True  # Concatenate trials for training

classifier = SsvepFbCcaClassifier(subset=[])
classifier.set_ssvep_settings(
    fsample=256.0,
    n_harmonics=n_harmonics,  # Number of harmonics to use for SSVEP classification
    target_freqs=target_frequencies,
    n_samples=nsamples,
    filter_bank=fb_cutoffs,
    concatenate_trials=concatenate_trials,
)

2025-04-09 15:51:24 - INFO - bci_essentials.classification.generic_classifier : Initializing the classifier


# Set up BCIController

In [4]:
test_ssvep = BciController(classifier, eeg_source, marker_source, paradigm, data_tank)

test_ssvep.setup(
    online=False,
    train_complete=True,  # Set to True to run the classifier on the entire dataset
)

2025-04-09 15:51:24 - INFO - bci_essentials.bci_controller : g.USBamp-1
2025-04-09 15:51:24 - INFO - bci_essentials.bci_controller : ['Fz', 'F4', 'F8', 'C3', 'Cz', 'C4', 'T8', 'P3', 'P4', 'P7', 'PO7', 'P8', 'PO8', 'O1', 'Oz', 'O2']


# Run

In [5]:
# Create a string buffer to capture log output
log_stream = StringIO()
handler = logging.StreamHandler(log_stream)
logger = logging.getLogger()
logger.addHandler(handler)

# Run the test
test_ssvep.run()

# Get log output as string
log_output = log_stream.getvalue()

# Extract true and predicted labels using regex
y_true = []
y_pred = []

for line in log_output.split('\n'):
    # Extract true labels
    if "Cue Object : " in line:
        true_label = int(re.search(r"Cue Object : (\d+)", line).group(1))
        y_true.append(true_label)
    
    # Extract predicted labels
    if "Predicted label: " in line:
        pred_label = int(re.search(r"Predicted label: (\d+)", line).group(1))
        y_pred.append(pred_label)

clear_output()

# Convert lists to numpy arrays
y_true = np.array(y_true[:-1])
y_pred = np.array(y_pred)

# Calculate and display accuracy
accuracy = np.mean(y_true == y_pred)
print(f"Accuracy: {accuracy:.2%}")

# Create confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

# Print detailed classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))
print(f"Number of samples: {len(y_true)}")
print(f"True labels: {y_true}")
print(f"Predicted labels: {y_pred}")

Accuracy: 26.44%

Confusion Matrix:
[[17  0  6  0]
 [16  0  7  0]
 [12  1  6  0]
 [15  1  6  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.28      0.74      0.41        23
           1       0.00      0.00      0.00        23
           2       0.24      0.32      0.27        19
           3       0.00      0.00      0.00        22

    accuracy                           0.26        87
   macro avg       0.13      0.26      0.17        87
weighted avg       0.13      0.26      0.17        87

Number of samples: 87
True labels: [2 1 2 1 2 3 1 2 0 0 3 0 2 1 0 0 1 1 0 3 3 3 1 1 3 1 2 1 3 3 1 2 3 0 3 3 3
 3 2 3 3 1 1 2 1 2 0 0 2 1 0 0 1 0 0 3 0 1 2 1 2 0 1 0 2 1 2 0 1 3 3 0 0 2
 3 0 3 1 2 0 2 0 2 3 0 3 1]
Predicted labels: [2 0 2 2 0 0 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 1 2 0 2 0 2 0 0 0 0 0 0 2
 0 1 2 0 0 0 0 0 0 0 2 2 0 0 0 2 0 0 2 0 0 0 0 2 0 2 0 0 0 0 2 0 2 0 0 2 0
 0 2 0 2 2 0 2 0 0 2 2 0 2]


c:\Users\admin\miniconda3\envs\bessy\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\admin\miniconda3\envs\bessy\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\admin\miniconda3\envs\bessy\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Get targets

In [6]:
markers = test_ssvep.marker_data
targets = []

for marker in markers:
    if "Cue " in marker:
        targets.append(int(marker.split(" ")[-1]))

print(targets)

#remove last target (this is just because I stopped Eli's data collectoin in the middle of a trial so there is a cue for the last trial but no actual data/markers from the trial)
targets.pop()
print(targets)


[2, 1, 2, 1, 2, 3, 1, 2, 0, 0, 3, 0, 2, 1, 0, 0, 1, 1, 0, 3, 3, 3, 1, 1, 3, 1, 2, 1, 3, 3, 1, 2, 3, 0, 3, 3, 3, 3, 2, 3, 3, 1, 1, 2, 1, 2, 0, 0, 2, 1, 0, 0, 1, 0, 0, 3, 0, 1, 2, 1, 2, 0, 1, 0, 2, 1, 2, 0, 1, 3, 3, 0, 0, 2, 3, 0, 3, 1, 2, 0, 2, 0, 2, 3, 0, 3, 1, 0]
[2, 1, 2, 1, 2, 3, 1, 2, 0, 0, 3, 0, 2, 1, 0, 0, 1, 1, 0, 3, 3, 3, 1, 1, 3, 1, 2, 1, 3, 3, 1, 2, 3, 0, 3, 3, 3, 3, 2, 3, 3, 1, 1, 2, 1, 2, 0, 0, 2, 1, 0, 0, 1, 0, 0, 3, 0, 1, 2, 1, 2, 0, 1, 0, 2, 1, 2, 0, 1, 3, 3, 0, 0, 2, 3, 0, 3, 1, 2, 0, 2, 0, 2, 3, 0, 3, 1]
